In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/hananfath/criminal-dataset/criminals.csv
/kaggle/input/datasets/hananfath/emp-name-email/enron_emp.csv
/kaggle/input/datasets/hananfath/enron-data-full/enron_full_preprocessed_v2.parquet


In [2]:
!pip uninstall -y numpy scipy gensim node2vec
!pip install numpy==1.26.4 scipy==1.12.0 gensim==4.3.2 node2vec==0.5.0

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.12.0
Uninstalling scipy-1.12.0:
  Successfully uninstalled scipy-1.12.0
Found existing installation: gensim 4.3.2
Uninstalling gensim-4.3.2:
  Successfully uninstalled gensim-4.3.2
Found existing installation: node2vec 0.5.0
Uninstalling node2vec-0.5.0:
  Successfully uninstalled node2vec-0.5.0
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached gensim-4.3.2-cp312-cp312-linux_x86_64.whl
  Using cached node2vec-0.5.0-py3-none-any.whl.metadata (849 bytes)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached scipy-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.8 MB)
Using cached node2vec-0.5.0-py3-non

In [3]:
import numpy as np
import scipy
import gensim

print(np.__version__)
print(scipy.__version__)
print(gensim.__version__)

from node2vec import Node2Vec
print("Node2Vec OK")

1.26.4
1.12.0
4.3.2
Node2Vec OK


In [4]:
import pandas as pd
import numpy as np

In [5]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.3 MB/s eta 0:00:00


In [6]:
!pip install node2vec

In [7]:
import torch
print(torch.__version__)

2.10.0+cu128


In [8]:
import pandas as pd

df1 = pd.read_parquet(
    "/kaggle/input/datasets/hananfath/enron-data-full/enron_full_preprocessed_v2.parquet"
)

print(df1.shape)
df1.head()

(327796, 21)


,Folder-Name,Date,From,To,X-From,X-To,X-Folder,X-FileName,Body,POI-Present,...,Unique-Mails-From-Sender,Low-Comm,Contains-Reply-Forwards,Label,Year,Processed_Body,From_Name,To_Name,From_Name_X,To_Name_X
0,arnold-j,Nov-2000,msagel@home.com,jarnold@enron.com,"""Mark Sagel"" <msagel@home.com>","""John Arnold"" <jarnold@enron.com>",Notes inbox,Jarnold.nsf,Status John: I'm not really sure what happened...,False,...,18.0,False,False,0,2000,status john really sure happened us impression...,"""mark sagel""","""john arnold""","""Mark Sagel""","""John Arnold"""
1,arnold-j,Dec-2000,slafontaine@globalp.com,john.arnold@enron.com,slafontaine@globalp.com,John.Arnold@enron.com,Notes inbox,Jarnold.nsf,summer inverses i suck-hope youve made more mo...,False,...,4.0,True,False,0,2000,summer inverses suck hope youve made money nat...,slafontaine,john arnold,slafontaine,John.Arnold
2,arnold-j,May-2001,iceoperations@intcx.com,icehelpdesk@intcx.com,ICE Operations <ICEOperations@intcx.com>,"**ICEHELPDESK <**ICEHELPDESK@intcx.com>, **Int...",Notes inbox,Jarnold.nsf,"The WTI Bullet swap contracts Hi, Following th...",False,...,3.0,True,False,0,2001,wti bullet swap contracts hi following e mail ...,ice operations,"**icehelpdesk , **internal marketing",ICE Operations,"**ICEHELPDESK , **Internal Marketing"
3,arnold-j,Dec-2000,klarnold@flash.net,john.arnold@enron.com,Karen Arnold <klarnold@flash.net>,john.arnold@enron.com,Notes inbox,Jarnold.nsf,NYTimes.com Article: Suspended Rabbi Quits Sem...,False,...,9.0,False,True,0,2000,nytimes article suspended rabbi quits seminary...,karen arnold,john arnold,Karen Arnold,john.arnold
4,arnold-j,May-2001,soblander@carrfut.com,soblander@carrfut.com,SOblander@carrfut.com,soblander@carrfut.com,Notes inbox,Jarnold.nsf,daily charts and matrices as hot links 5/15 Th...,False,...,352.0,False,False,0,2001,daily charts matrices hot links 5 15 informati...,soblander,soblander,SOblander,soblander


In [9]:
df2 = pd.read_csv("/kaggle/input/datasets/hananfath/criminal-dataset/criminals.csv")

In [10]:
df2["name"] = (
    df2["name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df2.head()

,num,name,label
0,2,andrew fastow,1
1,6,ben glisan,1
2,9,boyle dan,1
3,11,brown james,1
4,12,calger christopher,1


In [11]:
df3 = pd.read_csv("/kaggle/input/datasets/hananfath/emp-name-email/enron_emp.csv")

In [12]:
import pandas as pd
import re

df3 = pd.read_csv(
    "/kaggle/input/datasets/hananfath/emp-name-email/enron_emp.csv",
    sep="\t"
)

df3 = df3.rename(columns={"Unnamed: 5": "email4"})

email_map = {}

for _, row in df3.iterrows():
    name = row["name"]
    for col in ["email1", "email2", "email3", "email4"]:
        email = row[col]
        if pd.notna(email):
            email_map[email.strip().lower()] = name


def clean_name(x):
    if not isinstance(x, str) or not x:
        return ""
    
    x = re.sub(r"<.*?>", "", x)
    x = re.sub(r"/.*", "", x)
    x = re.sub(r"@[a-zA-Z0-9.-]+", "", x)
    
    return x.strip()


df1["From"] = df1["From"].fillna("").astype(str).str.strip().str.lower()
df1["To"] = df1["To"].fillna("").astype(str).str.strip().str.lower()

df1["From_Name"] = df1["From"].map(email_map)
df1["To_Name"] = df1["To"].map(email_map)

df1["X-From"] = df1["X-From"].fillna("")
df1["From_Name_X"] = df1["X-From"].apply(clean_name)

df1["From_Name"] = df1["From_Name"].fillna(df1["From_Name_X"])

df1["From_Name"] = df1["From_Name"].replace("", pd.NA)
df1["From_Name"] = df1["From_Name"].fillna(
    df1["From"].str.split("@").str[0]
)

df1["To_Name"] = df1["To_Name"].replace("", pd.NA)
df1["To_Name"] = df1["To_Name"].fillna(
    df1["To"].str.split("@").str[0]
)

df1["From_Name"] = (
    df1["From_Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df1["To_Name"] = (
    df1["To_Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df1 = df1.drop_duplicates(
    subset=["From_Name", "To_Name", "Processed_Body"]
)

print(df1.shape)
print("Rows with multiple recipients in To_Name:",
      df1["To_Name"].astype(str).str.contains(",", regex=False).sum())
print("Blank From_Name:",
      (df1["From_Name"].str.strip() == "").sum())
print("Blank To_Name:",
      (df1["To_Name"].str.strip() == "").sum())

(327696, 21)
Rows with multiple recipients in To_Name: 0
Blank From_Name: 0
Blank To_Name: 0


In [13]:
criminal_names = set(df2["name"])

from_matches = set(df1["From_Name"]) & criminal_names
to_matches = set(df1["To_Name"]) & criminal_names

all_matches = from_matches | to_matches

print("Total criminals in df2:", len(criminal_names))
print("Matched criminals in df1:", len(all_matches))
print(sorted(list(all_matches)))

Total criminals in df2: 25
Matched criminals in df1: 25
['andrew fastow', 'ben glisan', 'boyle dan', 'brown james', 'calger christopher', 'colwell wesley', 'david delainey', 'despain timothy', 'fastow lea', 'hirko joseph', 'jeffery skilling', 'john forney', 'ken rice', 'kenneth lay', 'kevin hannon', 'kevin howard', 'lawyer larry', 'mark koenig', 'mike krautz', 'paula rieker', 'ray bowen', 'richard causey', 'richter jeffrey', 'shelby rex', 'timothy belden']


In [14]:
criminal_nodes = set(df2["name"])

print("Criminal nodes:", len(criminal_nodes))

Criminal nodes: 25


**1-HOP**

In [15]:
direct_contacts = set()

for _, row in df1.iterrows():

    sender = row["From_Name"]
    receiver = row["To_Name"]

    if sender in criminal_nodes:
        direct_contacts.add(receiver)

    if receiver in criminal_nodes:
        direct_contacts.add(sender)

print("Direct contacts:", len(direct_contacts))

Direct contacts: 2710


In [16]:
selected_nodes = criminal_nodes.union(direct_contacts)

print("Total selected nodes:", len(selected_nodes))

Total selected nodes: 2713


In [17]:
df_subgraph = df1[
    (df1["From_Name"].isin(selected_nodes)) &
    (df1["To_Name"].isin(selected_nodes))
].copy()

print(df_subgraph.shape)

(97547, 21)


In [18]:
print("Nodes in subgraph:",
      len(
          set(df_subgraph["From_Name"])
          |
          set(df_subgraph["To_Name"])
      ))

print("Emails/Edges retained:",
      len(df_subgraph))

Nodes in subgraph: 2713
Emails/Edges retained: 97547


In [19]:
import networkx as nx

In [20]:
edge_counts_a1 = (
    df_subgraph.groupby(["From_Name", "To_Name"])
    .size()
    .reset_index(name="weight")
)

G_a1 = nx.from_pandas_edgelist(
    edge_counts_a1,
    source="From_Name",
    target="To_Name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

print("Approach 1 Graph Nodes:", G_a1.number_of_nodes())
print("Approach 1 Graph Edges:", G_a1.number_of_edges())

Approach 1 Graph Nodes: 2713
Approach 1 Graph Edges: 19278


In [21]:
print("Self-loops:", nx.number_of_selfloops(G_a1))

Self-loops: 81


In [22]:
G_a1.remove_edges_from(nx.selfloop_edges(G_a1))

print("Self-loops after removal:",
      nx.number_of_selfloops(G_a1))

Self-loops after removal: 0


**2-HOP**

In [23]:
hop1 = set()

for _, row in df1.iterrows():

    sender = row["From_Name"]
    receiver = row["To_Name"]

    if sender in criminal_nodes:
        hop1.add(receiver)

    if receiver in criminal_nodes:
        hop1.add(sender)

print("1-hop neighbors:", len(hop1))

1-hop neighbors: 2710


In [24]:
hop2 = set()

for _, row in df1.iterrows():

    sender = row["From_Name"]
    receiver = row["To_Name"]

    if sender in hop1:
        hop2.add(receiver)

    if receiver in hop1:
        hop2.add(sender)

print("2-hop neighbors:", len(hop2))

2-hop neighbors: 21662


In [25]:
selected_nodes_a2 = (
    criminal_nodes
    .union(hop1)
    .union(hop2)
)

print("Total selected nodes:", len(selected_nodes_a2))

Total selected nodes: 21667


In [26]:
df_subgraph_a2 = df1[
    (df1["From_Name"].isin(selected_nodes_a2)) &
    (df1["To_Name"].isin(selected_nodes_a2))
].copy()

print(df_subgraph_a2.shape)

(285638, 21)


In [27]:
import networkx as nx

edge_counts_a2 = (
    df_subgraph_a2.groupby(["From_Name", "To_Name"])
    .size()
    .reset_index(name="weight")
)

G_a2 = nx.from_pandas_edgelist(
    edge_counts_a2,
    source="From_Name",
    target="To_Name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

print("Approach 2 Graph Nodes:", G_a2.number_of_nodes())
print("Approach 2 Graph Edges:", G_a2.number_of_edges())
print("Self-loops:", nx.number_of_selfloops(G_a2))

Approach 2 Graph Nodes: 21667
Approach 2 Graph Edges: 75373
Self-loops: 142


In [28]:
G_a2.remove_edges_from(nx.selfloop_edges(G_a2))

print("Self-loops after removal:",
      nx.number_of_selfloops(G_a2))

Self-loops after removal: 0


**3-HOP**

In [29]:
# =====================================================
# Approach 3: Criminal Nodes + 1-Hop + 2-Hop + 3-Hop
# =====================================================

# -----------------------------
# 1. Find 1-hop neighbors
# -----------------------------

hop1 = set()

for _, row in df1.iterrows():

    sender = row["From_Name"]
    receiver = row["To_Name"]

    if sender in criminal_nodes:
        hop1.add(receiver)

    if receiver in criminal_nodes:
        hop1.add(sender)

print("1-hop neighbors:", len(hop1))


# -----------------------------
# 2. Find 2-hop neighbors
# -----------------------------

hop2 = set()

for _, row in df1.iterrows():

    sender = row["From_Name"]
    receiver = row["To_Name"]

    if sender in hop1:
        hop2.add(receiver)

    if receiver in hop1:
        hop2.add(sender)

print("2-hop neighbors:", len(hop2))


# -----------------------------
# 3. Find 3-hop neighbors
# -----------------------------

hop3 = set()

for _, row in df1.iterrows():

    sender = row["From_Name"]
    receiver = row["To_Name"]

    if sender in hop2:
        hop3.add(receiver)

    if receiver in hop2:
        hop3.add(sender)

print("3-hop neighbors:", len(hop3))


# -----------------------------
# 4. Combine selected nodes
# -----------------------------

selected_nodes_a3 = (
    criminal_nodes
    .union(hop1)
    .union(hop2)
    .union(hop3)
)

print("Total selected nodes:", len(selected_nodes_a3))


# -----------------------------
# 5. Create 3-hop subgraph dataframe
# -----------------------------

df_subgraph_a3 = df1[
    (df1["From_Name"].isin(selected_nodes_a3)) &
    (df1["To_Name"].isin(selected_nodes_a3))
].copy()

print("3-hop subgraph dataframe shape:", df_subgraph_a3.shape)


# -----------------------------
# 6. Create weighted edge list
# -----------------------------

import networkx as nx

edge_counts_a3 = (
    df_subgraph_a3.groupby(["From_Name", "To_Name"])
    .size()
    .reset_index(name="weight")
)

print("3-hop edge list shape:", edge_counts_a3.shape)


# -----------------------------
# 7. Build directed weighted graph
# -----------------------------

G_a3 = nx.from_pandas_edgelist(
    edge_counts_a3,
    source="From_Name",
    target="To_Name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

print("Approach 3 Graph Nodes:", G_a3.number_of_nodes())
print("Approach 3 Graph Edges:", G_a3.number_of_edges())
print("Self-loops:", nx.number_of_selfloops(G_a3))


# -----------------------------
# 8. Remove self-loops
# -----------------------------

G_a3.remove_edges_from(
    nx.selfloop_edges(G_a3)
)

print("Self-loops after removal:",
      nx.number_of_selfloops(G_a3))

1-hop neighbors: 2710
2-hop neighbors: 21662
3-hop neighbors: 31881
Total selected nodes: 31887
3-hop subgraph dataframe shape: (317601, 21)
3-hop edge list shape: (88390, 3)
Approach 3 Graph Nodes: 31887
Approach 3 Graph Edges: 88390
Self-loops: 158
Self-loops after removal: 0


**FEATURE ENGINEERING**

In [30]:
import pandas as pd
import networkx as nx
from node2vec import Node2Vec

# =====================================================
# Function: Feature Engineering with Sent + Received Features
# =====================================================

def create_node_features(graph, subgraph_df, graph_name):
    
    print(f"\n========== {graph_name} Feature Engineering ==========")

    # -------------------------------
    # 1. All graph nodes
    # -------------------------------

    all_nodes_df = pd.DataFrame({
        "Node": list(graph.nodes())
    })

    # -------------------------------
    # 2. Sent Email Features
    # -------------------------------

    total_sent = (
        subgraph_df.groupby("From_Name")
        .size()
        .reset_index(name="total_emails_sent")
    )

    total_sent.columns = ["Node", "total_emails_sent"]

    suspicious_sent = (
        subgraph_df.groupby("From_Name")["Label"]
        .sum()
        .reset_index(name="suspicious_emails_sent")
    )

    suspicious_sent.columns = ["Node", "suspicious_emails_sent"]

    sent_features = total_sent.merge(
        suspicious_sent,
        on="Node",
        how="left"
    )

    # -------------------------------
    # 3. Received Email Features
    # -------------------------------

    total_received = (
        subgraph_df.groupby("To_Name")
        .size()
        .reset_index(name="total_emails_received")
    )

    total_received.columns = ["Node", "total_emails_received"]

    suspicious_received = (
        subgraph_df.groupby("To_Name")["Label"]
        .sum()
        .reset_index(name="suspicious_emails_received")
    )

    suspicious_received.columns = ["Node", "suspicious_emails_received"]

    received_features = total_received.merge(
        suspicious_received,
        on="Node",
        how="left"
    )

    # -------------------------------
    # 4. Merge Sent + Received Features
    # -------------------------------

    email_features = all_nodes_df.merge(
        sent_features,
        on="Node",
        how="left"
    )

    email_features = email_features.merge(
        received_features,
        on="Node",
        how="left"
    )

    email_features = email_features.fillna(0)

    email_features["total_email_activity"] = (
        email_features["total_emails_sent"] +
        email_features["total_emails_received"]
    )

    email_features["total_suspicious_email_activity"] = (
        email_features["suspicious_emails_sent"] +
        email_features["suspicious_emails_received"]
    )

    email_features["sent_suspicious_ratio"] = (
        email_features["suspicious_emails_sent"] /
        email_features["total_emails_sent"].replace(0, 1)
    )

    email_features["received_suspicious_ratio"] = (
        email_features["suspicious_emails_received"] /
        email_features["total_emails_received"].replace(0, 1)
    )

    email_features["overall_suspicious_ratio"] = (
        email_features["total_suspicious_email_activity"] /
        email_features["total_email_activity"].replace(0, 1)
    )

    print("Email Features:", email_features.shape)

    # -------------------------------
    # 5. Centrality Features
    # -------------------------------

    degree_centrality = nx.degree_centrality(graph)

    degree_df = pd.DataFrame({
        "Node": list(degree_centrality.keys()),
        "Degree_Centrality": list(degree_centrality.values())
    })

    pagerank = nx.pagerank(
        graph,
        alpha=0.85,
        weight="weight"
    )

    pagerank_df = pd.DataFrame({
        "Node": list(pagerank.keys()),
        "PageRank": list(pagerank.values())
    })

    eigenvector = nx.eigenvector_centrality(
        graph,
        max_iter=1000,
        weight="weight"
    )

    eigenvector_df = pd.DataFrame({
        "Node": list(eigenvector.keys()),
        "Eigenvector_Centrality": list(eigenvector.values())
    })

    # Approximate betweenness to reduce time
    if graph.number_of_nodes() <= 3000:
        betweenness = nx.betweenness_centrality(
            graph,
            normalized=True
        )
    else:
        betweenness = nx.betweenness_centrality(
            graph,
            k=1000,
            normalized=True,
            seed=42
        )

    betweenness_df = pd.DataFrame({
        "Node": list(betweenness.keys()),
        "Betweenness_Centrality": list(betweenness.values())
    })

    centrality_df = degree_df.merge(
        pagerank_df,
        on="Node"
    ).merge(
        eigenvector_df,
        on="Node"
    ).merge(
        betweenness_df,
        on="Node"
    )

    print("Centrality Features:", centrality_df.shape)

    # -------------------------------
    # 6. Merge Email + Centrality
    # -------------------------------

    node_features_basic = email_features.merge(
        centrality_df,
        on="Node",
        how="inner"
    )

    print("Basic Features:", node_features_basic.shape)

    # -------------------------------
    # 7. Node2Vec Embeddings
    # -------------------------------

    node2vec = Node2Vec(
        graph,
        dimensions=64,
        walk_length=20,
        num_walks=10,
        workers=1,
        weight_key="weight",
        seed=42
    )

    model = node2vec.fit(
        window=10,
        min_count=1
    )

    embeddings = []

    for node in graph.nodes():
        embeddings.append(
            [node] + list(model.wv[str(node)])
        )

    embedding_df = pd.DataFrame(
        embeddings,
        columns=["Node"] + [f"node2vec_{i}" for i in range(64)]
    )

    print("Node2Vec Embeddings:", embedding_df.shape)

    # -------------------------------
    # 8. Final Feature Table
    # -------------------------------

    node_features_final = node_features_basic.merge(
        embedding_df,
        on="Node",
        how="inner"
    )

    # Add labels
    node_features_final["Label"] = (
        node_features_final["Node"]
        .isin(criminal_nodes)
        .astype(int)
    )

    print("Final Features:", node_features_final.shape)
    print("Label Count:")
    print(node_features_final["Label"].value_counts())

    return node_features_basic, node_features_final

Louvain

In [36]:
!pip install python-louvain

**TOPIC MODELLING**

In [54]:
print(df1.columns.tolist())
print(df1.shape)

['Folder-Name', 'Date', 'From', 'To', 'X-From', 'X-To', 'X-Folder', 'X-FileName', 'Body', 'POI-Present', 'Sender-Type', 'Unique-Mails-From-Sender', 'Low-Comm', 'Contains-Reply-Forwards', 'Label', 'Year', 'Processed_Body', 'From_Name', 'To_Name', 'From_Name_X', 'To_Name_X']
(327696, 21)


In [55]:
topic_df = df1[
    ["From_Name", "To_Name", "Processed_Body", "Label"]
].copy()

topic_df = topic_df.dropna(
    subset=["From_Name", "To_Name", "Processed_Body"]
)

topic_df = topic_df[
    topic_df["Processed_Body"].astype(str).str.strip() != ""
].copy()

print(topic_df.shape)
print(topic_df.isnull().sum())
print(topic_df["Label"].value_counts())

(327696, 4)
From_Name         0
To_Name           0
Processed_Body    0
Label             0
dtype: int64
Label
0    326013
1      1683
Name: count, dtype: int64


In [56]:
topic_df["word_count"] = (
    topic_df["Processed_Body"]
    .astype(str)
    .str.split()
    .str.len()
)

print(topic_df["word_count"].describe())

count    327696.000000
mean        115.723475
std         778.821661
min           1.000000
25%          14.000000
50%          30.000000
75%          69.000000
max      139681.000000
Name: word_count, dtype: float64


In [57]:
print(topic_df.shape)

criminal_emails = topic_df[topic_df["Label"] == 1]
print("Criminal emails:", criminal_emails.shape)

normal_emails = topic_df[topic_df["Label"] == 0]
print("Normal emails:", normal_emails.shape)

(327696, 5)
Criminal emails: (1683, 5)
Normal emails: (326013, 5)


In [58]:
print("Unique Senders:",
      topic_df["From_Name"].nunique())

print("Unique Receivers:",
      topic_df["To_Name"].nunique())

all_people = set(topic_df["From_Name"]) | set(topic_df["To_Name"])

print("Total Unique People:",
      len(all_people))

Unique Senders: 20494
Unique Receivers: 18124
Total Unique People: 37244


In [59]:
print("Total Emails:",
      len(topic_df))

print("Unique Processed Bodies:",
      topic_df["Processed_Body"].nunique())

Total Emails: 327696
Unique Processed Bodies: 214678


In [60]:
!pip install bertopic sentence-transformers umap-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<

In [61]:
import sentence_transformers
import bertopic
import umap
import sklearn

print("sentence_transformers OK")
print("bertopic OK")
print("umap OK")
print("sklearn OK")

2026-06-12 04:08:35.346835: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781237315.523584     246 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781237315.572961     246 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781237316.001058     246 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781237316.001085     246 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781237316.001087     246 computation_placer.cc:177] computation placer alr

sentence_transformers OK
bertopic OK
umap OK
sklearn OK


In [62]:
clean_texts = (
    topic_df["Processed_Body"]
    .astype(str)
    .tolist()
)

print("Number of texts:", len(clean_texts))
print("First text sample:")
print(clean_texts[0][:500])

Number of texts: 327696
First text sample:
status john really sure happened us impression visit houston enter trial agreement advisory work somehow never occurred say something wrong screw know blown whole thing still hope interested trying create arrangement courtesy report past weekend longer interested work please tell best wishes mark sagel psytech analytics 410 308 0245 energy2000 1112 doc


In [63]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [64]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    clean_texts,
    show_progress_bar=True,
    batch_size=64
)

print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5121 [00:00<?, ?it/s]

Embeddings shape: (327696, 384)


In [65]:
print("Embeddings shape:", embeddings.shape)
print(type(embeddings))

Embeddings shape: (327696, 384)
<class 'numpy.ndarray'>


In [66]:
from umap import UMAP

umap_model = UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.1,
    metric="cosine",
    low_memory=True,
    random_state=42
)

print(umap_model)

UMAP(metric='cosine', n_components=5, n_neighbors=10, random_state=42)


In [67]:
from sklearn.cluster import KMeans

cluster_model = KMeans(
    n_clusters=10,
    random_state=42
)

print(cluster_model)

KMeans(n_clusters=10, random_state=42)


In [68]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    min_df=2,
    ngram_range=(1, 2),
    max_features=5000
)

print(vectorizer_model)

CountVectorizer(max_features=5000, min_df=2, ngram_range=(1, 2))


In [69]:
from bertopic import BERTopic

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True,
    top_n_words=10
)

print(topic_model)

BERTopic(calculate_probabilities=True, ctfidf_model=ClassTfidfTransformer(...), embedding_model=None, hdbscan_model=KMeans(...), language=english, low_memory=False, min_topic_size=10, n_gram_range=(1, 1), nr_topics=None, representation_model=None, seed_topic_list=None, top_n_words=10, umap_model=UMAP(...), vectorizer_model=CountVectorizer(...), verbose=True, zeroshot_min_similarity=0.7, zeroshot_topic_list=None)


In [70]:
import psutil

print("RAM (GB):",
      round(psutil.virtual_memory().total / 1024**3, 2))

RAM (GB): 31.35


In [71]:
topics, probs = topic_model.fit_transform(
    clean_texts,
    embeddings
)

print("Number of topic labels:", len(topics))

if probs is not None:
    print("Probability matrix shape:", probs.shape)
else:
    print("Probabilities not generated")

2026-06-12 04:14:29,109 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-12 05:13:30,558 - BERTopic - Dimensionality - Completed ✓
2026-06-12 05:13:30,565 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-12 05:13:31,183 - BERTopic - Cluster - Completed ✓
2026-06-12 05:13:31,240 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-12 05:14:46,962 - BERTopic - Representation - Completed ✓


Number of topic labels: 327696
Probabilities not generated


In [72]:
topic_info = topic_model.get_topic_info()

print(topic_info.head(15))

   Topic   Count                               Name  \
0      0  115403            0_please_thanks_2001_00   
1      1   84861                  1_url_09_image_01   
2      2   53404                2_09_20_company_url   
3      3   20687            3_20_power_energy_state   
4      4    9470                 4_09_09 09_epmi_00   
5      5    9292  5_please_type_credit_credit watch   
6      6    9268              6_20_power_said_error   
7      7    8669             7_20_said_power_energy   
8      8    8411                8_3d 3d_3d_20_power   
9      9    8231              9_20_said_power_state   

                                      Representation  \
0  [please, thanks, 2001, 00, 713, agreement, att...   
1  [url, 09, image, 01, 00, one, 10, free, mail, ...   
2  [09, 20, company, url, 09 09, said, new, pleas...   
3  [20, power, energy, state, california, said, g...   
4  [09, 09 09, epmi, 00, total, term, 50, transac...   
5  [please, type, credit, credit watch, x3, trans...   
6 

In [73]:
for topic_id in range(10):
    print("\n" + "="*60)
    print(f"TOPIC {topic_id}")
    print("="*60)

    print(topic_model.get_topic(topic_id))


TOPIC 0
[('please', 0.03045828550758827), ('thanks', 0.02280149210299886), ('2001', 0.021475872824884928), ('00', 0.021236807350111436), ('713', 0.019200340656965017), ('agreement', 0.018564431748827884), ('attached', 0.01824926373932521), ('would', 0.017562845877210773), ('gas', 0.017549443854549717), ('know', 0.01667280641687719)]

TOPIC 1
[('url', 0.05343847693417909), ('09', 0.028415300278329145), ('image', 0.02522766495052017), ('01', 0.02052981457350818), ('00', 0.020171766409729685), ('one', 0.018265420766081068), ('10', 0.017711293215766612), ('free', 0.017530977919907955), ('mail', 0.016960867930329495), ('get', 0.016895195254567782)]

TOPIC 2
[('09', 0.025481869008437717), ('20', 0.02172665153527362), ('company', 0.0216610313390576), ('url', 0.021544442408753858), ('09 09', 0.018751660771367345), ('said', 0.016756820351427375), ('new', 0.015617575904966952), ('please', 0.015107761536549377), ('meeting', 0.01492521249508265), ('would', 0.014636890641672458)]

TOPIC 3
[('20', 

In [74]:
docs = topic_model.get_representative_docs(3)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:500])   # only first 500 characters

Number of representative docs: 3

----- Document 1 -----
davis press releases today 1 announces two peakers 2 agrees sen perata energy conservation rebate plan enacts exec order immediate release 03 13 2001 11 30 20 governor davis unveils 20 20 energy rebate 20 program summer 2001 sacramento 20 california ratepayers receive 20 percent rebate summer 20 electric bill cut back electricity 20 percent las 20 summer levels new initiative unveiled governor davis today 20 20 program created governor executive order 20 designed help state avoid likelihood blac

----- Document 2 -----
energy issues thurs please see following articles sac bee thurs 5 24 davis push backup diesel sac bee thurs 5 24 bush davis meeting set energy crisis sac bee thurs 5 24 energy digest gop unveils plan help utility sac bee thurs 5 24 escape blame crisis poll sac bee thurs 5 24 new views emerging power elected officials support concept planned blackouts sac bee thurs 5 24 californians priorities solving crisis 20 outl

In [75]:
topic_df["Topic"] = topics

print(topic_df[[
    "Label",
    "Topic"
]].head())

print("\nUnique Topics:")
print(sorted(topic_df["Topic"].unique()))

   Label  Topic
0      0      2
1      0      0
2      0      0
3      0      2
4      0      0

Unique Topics:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [76]:
topic_label_counts = pd.crosstab(
    topic_df["Topic"],
    topic_df["Label"]
)

topic_label_counts.columns = [
    "Normal",
    "Criminal"
]

print(topic_label_counts)

       Normal  Criminal
Topic                  
0      115261       142
1       83721      1140
2       53118       286
3       20654        33
4        9448        22
5        9251        41
6        9263         5
7        8662         7
8        8409         2
9        8226         5


In [77]:
topic_percentages = topic_label_counts.copy()

topic_percentages["Normal_%"] = (
    topic_percentages["Normal"]
    / topic_percentages["Normal"].sum()
    * 100
)

topic_percentages["Criminal_%"] = (
    topic_percentages["Criminal"]
    / topic_percentages["Criminal"].sum()
    * 100
)

topic_percentages["Difference"] = (
    topic_percentages["Criminal_%"]
    - topic_percentages["Normal_%"]
)

topic_percentages = topic_percentages.sort_values(
    "Difference",
    ascending=False
)

print(topic_percentages.round(2))

       Normal  Criminal  Normal_%  Criminal_%  Difference
Topic                                                    
1       83721      1140     25.68       67.74       42.06
2       53118       286     16.29       16.99        0.70
5        9251        41      2.84        2.44       -0.40
4        9448        22      2.90        1.31       -1.59
9        8226         5      2.52        0.30       -2.23
7        8662         7      2.66        0.42       -2.24
8        8409         2      2.58        0.12       -2.46
6        9263         5      2.84        0.30       -2.54
3       20654        33      6.34        1.96       -4.37
0      115261       142     35.35        8.44      -26.92


In [78]:
docs = topic_model.get_representative_docs(1)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
commissioner e reports perfect league 11 17 01 image 09 save 10 speak evil hear evil monkey bikes one hottest g ifts holiday 2001 enter coupon code mqj36gx6 checkou process receive discount offer expires november 16 2001 want win fantasy league fantasy football guides source fo r strategy player ratings scouting reports team reports projections must beginners fantasy veterans alike special eason price 9 99 going fast click save 05 gallon gas keeps car engine clean click apply online brought sponsorship bar receiving e reports signed cbs sportsline fantasy football customize resc hedule turn reports please click nfl reports player updates image latest nfl player news correll buckhalter rb phi lie downs updated 11 17 01 buckhalter suspended weekend game dallas one three eagles held game brian mitchell likely serve duce staley backup jamal lew rb bal free agent updated 11 17 01 according published reports l ewis suspended four games

In [79]:
docs = topic_model.get_representative_docs(5)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
credit watch list 9 5 00 attached newly revised credit watch listing personnel group included distribution please insure receive copy report add additional people distribution report error please contact veronica espinoza x6 6002 questions please contact bill bradford x3 3831 russell diamond x5 7095 brant reves x3 9897 veronica espinoza x6 6002 global credit hotline x3 1803

----- Document 2 -----
credit watch list 9 5 00 attached newly revised credit watch listing personnel group included distribution please insure receive copy report add additional people distribution report error please contact veronica espinoza x6 6002 questions please contact bill bradford x3 3831 russell diamond x5 7095 brant reves x3 9897 veronica espinoza x6 6002 global credit hotline x3 1803

----- Document 3 -----
credit watch list 9 5 00 attached newly revised credit watch listing personnel group included distribution please insure receive copy report 

In [80]:
docs = topic_model.get_representative_docs(4)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
november data 11 09 01 revised includes ice data total trades day 419 eol deals 11 01 2001 11 09 2001 enpower 11 01 2001 11 09 2001 desk total deals total mwh desk total deals total mwh epmi long term california 299 4 568 656 epmi long term california 32 1 183 354 epmi long term northwest 148 1 931 200 epmi long term northwest 68 2 105 562 epmi long term southwest 164 3 781 845 epmi long term southwest 188 7 199 574 epmi short term california 932 1 149 712 epmi short term california 339 885 297 epmi short term northwest 521 676 240 epmi short term northwest 220 449 461 epmi short term southwest 696 1 025 952 epmi short term southwest 354 1 490 794 real time 651 16 575 real time 211 18 787 grand total 3 411 13 150 180 grand total 1 412 13 332 830 eol deals 11 09 2001 11 09 2001 enpower 11 09 2001 11 09 2001 desk total deals total mwh desk total deals total mwh epmi long term california 67 1 068 800 epmi long term california 5 222 

In [81]:
docs = topic_model.get_representative_docs(2)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
jan 8 2002 reverse psychology today equity research columnist dave sterman examines recent public pronouncements nortel nt lucent lu analysts believe management companies might engaged subtle mind games also columnist john filar atwood explains focus cable tv industry consolidation ebidta growth finally readers might note qualcomm q figures prominently list ten recommended telecom stocks follow reading strong recommendation morgan stanley registering firm free research trial benefit available exclusively multex investor members like sponsored make 50 100 profits coming nasdaq surge learn secrets changewave investing strategy delivered 75 year growth since 1995including nasdaq crash 2000 2001 sign tobin smith free e mail seminar profiting change get session 1 url investment ideas broker reports top 10 free sponsored reports investment ideas 1 looking ahead analysts predict solid 2002 cable tv stocks consolidation ebitda growth yea

In [82]:
docs = topic_model.get_representative_docs(0)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
attached draft term sheet equity investment houston street discussed deal varies true quote deal houston street already closed several deals investors may dictate extent deal able get many terms conditions equity investment depend upon results due diligence investigation may depend upon terms conditions posting agreement please let know comments questions regarding attached travis mccullough north america corp 1400 smith street eb 3817 houston texas 77002 phone 713 853 1575 fax 713 646 3490

----- Document 2 -----
st mary letter agreement attached st mary letter agreement incorporating revisions relating vpp transaction marked copy showing changes last draft distributed please let know questions comments advise joan revisions ok thanks teresa g bushman north america corp 1400 smith street eb 3835a houston tx 77002 713 853 7895 fax 713 646 3393

----- Document 3 -----
credit inc eci consent gail attached form eci unanimous consent

In [83]:
docs = topic_model.get_representative_docs(6)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
energy issues please see attached articles sd union sun 6 3 poor find red tape federal energy aid sd union sun 6 3 top energy adviser confident gaining pact l 20 surplus sd union sun 6 3 energy firm ties enriched top advisers president 20 sd union sun 6 3 report state may little money left head 20 blackouts sd union sat 6 2 san onofre reactor back line sending power state 20 grid sd union sat 6 2 state files manage energy sd union sat 6 2 escondido running power plant process ex official 20 says 20 sd union sat 6 2 pg e bankruptcy judge challenge state regulation sd union sat 6 2 new majority leader says nay electricity price caps la times sun 6 3 watchdogs take hit state power ills la times sat 6 2 duke charged record price electricity 20 la times sat 6 2 incoming senate leader daschle lukewarm power price 20 caps la times sat 6 2 puc may trip bailout edison la times mon 6 4 better bankruptcy commentary sf chron mon 6 4 electric

In [84]:
docs = topic_model.get_representative_docs(7)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
energy issues please see following articles sac bee wed 5 16 double power punch rate hikes set blackout blues 20 ahead sac bee wed 5 16 energy council expects five times outages 20 forecast sac bee wed 5 16 generator environmental groups strike deal sac bee wed 5 16 blackouts may create shortage water state officia ls 20 warn supplies drinking fire hydrants vulnerable 20 pumps fail power outages sac bee wed 5 16 dan walters rate raise genuine conflict tightl 20 scripted political melodrama sac bee wed 5 16 budget reserve plan stirs fight gov davis back 20 drawing emergency fund others say could lead tax hikes 20 massive cuts sac bee wed 5 16 democrats lay energy plan leaders call caps 20 wholesale prices tax breaks oil gas production sac bee wed 5 16 judge names panel pg e case sd union wed 5 16 sdg e area spared big rate hikes sd union wed 5 16 state credit rating takes another hit energy 20 crisis sd union wed 5 16 house democr

In [85]:
docs = topic_model.get_representative_docs(9)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
energy issues please see following articles sac bee wed 7 18 lawmakers debate efforts save edison 20 competing bills range straight bailout purchase transmission 20 lines sac bee wed 7 18 pg e sues state seizure long term power contrac ts 20 sac bee wed 7 18 energy digest advocate attacks utility rescue plan sac bee wed 7 18 hour decision pay costly power past 20 future editorial sd union tues 7 17 utility sues state seeking millions seized power 20 contracts 20 la times wed 7 18 pg e sues state contracts sf chron wed 7 18 consumer prices energy costs moderate 20 sf chron wed 7 18 contra costa may upgrade 9 buildings 20 1 7 million earmarked conservation 20 sf chron wed 7 18 lawmakers devise rival bailout plans edison 20 push come alternative bankruptcy recess 20 sf chron wed 7 18 news briefs california power crisis 20 sf chron wed 7 18 state largest utility sues state seeking millions 20 power contracts seized governor 20 mercur

In [86]:
docs = topic_model.get_representative_docs(8)

print("Number of representative docs:", len(docs))

for i, doc in enumerate(docs[:3]):
    print(f"\n----- Document {i+1} -----")
    print(doc[:1000])

Number of representative docs: 3

----- Document 1 -----
news alert yhoo 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d requested news alert yhoo follows equityalert edit discontinue alerts please refer end please review notice disclaimer 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d paid advertisement 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d 3d looking emerging growth company exciting burgeoning industry search xraymedia otc bb xrmd currently assembling partners undertake

In [87]:
topic_summary = topic_df.groupby("Topic")["Label"].agg(
    ["count", "sum"]
).rename(columns={
    "count":"Total_Emails",
    "sum":"Criminal_Emails"
})

topic_summary["Criminal_Rate_%"] = (
    topic_summary["Criminal_Emails"]
    / topic_summary["Total_Emails"]
    * 100
)

topic_summary = topic_summary.sort_values(
    "Criminal_Rate_%",
    ascending=False
)

print(topic_summary.round(2))

       Total_Emails  Criminal_Emails  Criminal_Rate_%
Topic                                                
1             84861             1140             1.34
2             53404              286             0.54
5              9292               41             0.44
4              9470               22             0.23
3             20687               33             0.16
0            115403              142             0.12
7              8669                7             0.08
9              8231                5             0.06
6              9268                5             0.05
8              8411                2             0.02


**TOPIC MODELLING+GNN**

In [88]:
topic_df["Dominant_Topic"] = topics

In [89]:
topic_df[
    ["From_Name","To_Name","Dominant_Topic","Label"]
].head()

,From_Name,To_Name,Dominant_Topic,Label
0,"""mark sagel""",jarnold,2,0
1,slafontaine,john arnold,0,0
2,ice operations,icehelpdesk,0,0
3,karen arnold,john arnold,2,0
4,soblander,soblander,0,0


In [90]:
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data

# =====================================================
# 1. Add Dominant Topic to each email/document
# =====================================================

topic_df = topic_df.copy()

if "Dominant_Topic" not in topic_df.columns:
    topic_df["Dominant_Topic"] = topics

topic_df["_emb_pos"] = np.arange(len(topic_df))

print(topic_df[["From_Name", "To_Name", "Dominant_Topic", "Label"]].head())
print(topic_df["Dominant_Topic"].value_counts().sort_index())

        From_Name      To_Name  Dominant_Topic  Label
0    "mark sagel"      jarnold               2      0
1     slafontaine  john arnold               0      0
2  ice operations  icehelpdesk               0      0
3    karen arnold  john arnold               2      0
4       soblander    soblander               0      0
Dominant_Topic
0    115403
1     84861
2     53404
3     20687
4      9470
5      9292
6      9268
7      8669
8      8411
9      8231
Name: count, dtype: int64


In [91]:
def add_bert_topic_node_features(graph, feature_df, topic_df, embeddings, graph_name):

    print(f"\n========== Adding BERT + Topic Features: {graph_name} ==========")

    graph_nodes = set(graph.nodes())

    topic_subgraph_df = topic_df[
        (topic_df["From_Name"].isin(graph_nodes)) &
        (topic_df["To_Name"].isin(graph_nodes))
    ].copy()

    print("Topic subgraph emails:", topic_subgraph_df.shape)

    node_list = list(graph.nodes())
    node_to_idx = {node: idx for idx, node in enumerate(node_list)}

    # -------------------------------
    # BERT embedding aggregation
    # -------------------------------

    bert_dim = embeddings.shape[1]

    bert_sum = np.zeros(
        (len(node_list), bert_dim),
        dtype=np.float32
    )

    bert_count = np.zeros(
        len(node_list),
        dtype=np.float32
    )

    for _, row in topic_subgraph_df.iterrows():

        emb = embeddings[int(row["_emb_pos"])]

        sender = row["From_Name"]
        receiver = row["To_Name"]

        if sender in node_to_idx:
            idx = node_to_idx[sender]
            bert_sum[idx] += emb
            bert_count[idx] += 1

        if receiver in node_to_idx:
            idx = node_to_idx[receiver]
            bert_sum[idx] += emb
            bert_count[idx] += 1

    bert_count_safe = np.where(
        bert_count == 0,
        1,
        bert_count
    )

    bert_avg = bert_sum / bert_count_safe.reshape(-1, 1)

    bert_df = pd.DataFrame(
        bert_avg,
        columns=[f"bert_{i}" for i in range(bert_dim)]
    )

    bert_df.insert(0, "Node", node_list)

    print("BERT node features:", bert_df.shape)

    # -------------------------------
    # Topic distribution per node
    # -------------------------------

    num_topics = topic_df["Dominant_Topic"].nunique()

    topic_count = np.zeros(
        (len(node_list), num_topics),
        dtype=np.float32
    )

    for _, row in topic_subgraph_df.iterrows():

        topic = int(row["Dominant_Topic"])

        sender = row["From_Name"]
        receiver = row["To_Name"]

        if sender in node_to_idx:
            topic_count[node_to_idx[sender], topic] += 1

        if receiver in node_to_idx:
            topic_count[node_to_idx[receiver], topic] += 1

    topic_total = topic_count.sum(axis=1, keepdims=True)
    topic_total_safe = np.where(topic_total == 0, 1, topic_total)

    topic_ratio = topic_count / topic_total_safe

    topic_feature_df = pd.DataFrame(
        topic_ratio,
        columns=[f"topic_ratio_{i}" for i in range(num_topics)]
    )

    topic_feature_df.insert(0, "Node", node_list)

    topic_feature_df["Dominant_Node_Topic"] = np.argmax(
        topic_count,
        axis=1
    )

    print("Topic node features:", topic_feature_df.shape)

    # -------------------------------
    # Merge with old node features
    # -------------------------------

    enriched_features = (
        feature_df
        .merge(bert_df, on="Node", how="left")
        .merge(topic_feature_df, on="Node", how="left")
    )

    enriched_features = enriched_features.fillna(0)

    print("Final enriched features:", enriched_features.shape)
    print("Label count:")
    print(enriched_features["Label"].value_counts())

    return enriched_features

In [92]:
node_features_final_a1_topic = add_bert_topic_node_features(
    graph=G_a1,
    feature_df=node_features_final_a1_new,
    topic_df=topic_df,
    embeddings=embeddings,
    graph_name="1-Hop"
)

node_features_final_a3_topic = add_bert_topic_node_features(
    graph=G_a3,
    feature_df=node_features_final_a3_new,
    topic_df=topic_df,
    embeddings=embeddings,
    graph_name="3-Hop"
)


========== Adding BERT + Topic Features: 1-Hop ==========
Topic subgraph emails: (97547, 8)
BERT node features: (2713, 385)
Topic node features: (2713, 12)
Final enriched features: (2713, 474)
Label count:
Label
0    2688
1      25
Name: count, dtype: int64

========== Adding BERT + Topic Features: 3-Hop ==========
Topic subgraph emails: (317601, 8)
BERT node features: (31887, 385)
Topic node features: (31887, 12)
Final enriched features: (31887, 474)
Label count:
Label
0    31862
1       25
Name: count, dtype: int64


In [93]:
from torch_geometric.data import Data
import torch

def create_pyg_data_with_edge_features(
    graph,
    feature_df,
    topic_df
):

    node_list = list(graph.nodes())

    node_to_idx = {
        node: idx
        for idx, node in enumerate(node_list)
    }

    # --------------------------
    # Node Features
    # --------------------------

    feature_cols = [
        col for col in feature_df.columns
        if col not in ["Node", "Label"]
    ]

    X_df = (
        feature_df
        .set_index("Node")
        .loc[node_list, feature_cols]
    )

    X = torch.tensor(
        X_df.values,
        dtype=torch.float
    )

    y_df = (
        feature_df
        .set_index("Node")
        .loc[node_list, "Label"]
    )

    y = torch.tensor(
        y_df.values,
        dtype=torch.long
    )

    # --------------------------
    # Edge Index
    # --------------------------

    edge_index = []

    edge_attr = []

    topic_lookup = (
        topic_df
        .groupby(["From_Name", "To_Name"])
        ["Dominant_Topic"]
        .agg(lambda x: x.mode()[0])
        .to_dict()
    )

    for source, target in graph.edges():

        edge_index.append([
            node_to_idx[source],
            node_to_idx[target]
        ])

        topic = topic_lookup.get(
            (source, target),
            0
        )

        edge_attr.append([topic])

    edge_index = torch.tensor(
        edge_index,
        dtype=torch.long
    ).t().contiguous()

    edge_attr = torch.tensor(
        edge_attr,
        dtype=torch.float
    )

    data = Data(
        x=X,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y
    )

    return data

In [94]:
data_a1_topic = create_pyg_data_with_edge_features(
    G_a1,
    node_features_final_a1_topic,
    topic_df
)

data_a1_topic = create_masks(data_a1_topic)

print(data_a1_topic)

Data(x=[2713, 472], edge_index=[2, 19197], edge_attr=[19197, 1], y=[2713], train_mask=[2713], val_mask=[2713], test_mask=[2713])


In [95]:
data_a3_topic = create_pyg_data_with_edge_features(
    G_a3,
    node_features_final_a3_topic,
    topic_df
)

data_a3_topic = create_masks(data_a3_topic)

print(data_a3_topic)

Data(x=[31887, 472], edge_index=[2, 88232], edge_attr=[88232, 1], y=[31887], train_mask=[31887], val_mask=[31887], test_mask=[31887])


In [96]:
class GAT_Edge(torch.nn.Module):

    def __init__(self, num_features, hidden_channels, heads):
        super().__init__()

        self.conv1 = GATConv(
            num_features,
            hidden_channels,
            heads=heads,
            dropout=0.3,
            edge_dim=1
        )

        self.conv2 = GATConv(
            hidden_channels * heads,
            2,
            heads=1,
            dropout=0.3,
            edge_dim=1
        )

    def forward(self, x, edge_index, edge_attr):

        x = F.dropout(
            x,
            p=0.3,
            training=self.training
        )

        x = self.conv1(
            x,
            edge_index,
            edge_attr
        )

        x = F.elu(x)

        x = F.dropout(
            x,
            p=0.3,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index,
            edge_attr
        )

        return x

In [97]:
def train_and_evaluate_updated(
    model,
    data,
    model_name,
    use_edge_attr=False,
    class_weight_1=100.0,
    lr=0.01
):

    print("\n====================================")
    print(model_name)
    print("====================================")

    print(model)

    class_weights = torch.tensor(
        [0.5006, class_weight_1],
        dtype=torch.float
    )

    criterion = torch.nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=5e-4
    )

    def forward_model():
        if use_edge_attr:
            return model(
                data.x,
                data.edge_index,
                data.edge_attr
            )
        else:
            return model(
                data.x,
                data.edge_index
            )

    def train():
        model.train()
        optimizer.zero_grad()

        out = forward_model()

        loss = criterion(
            out[data.train_mask],
            data.y[data.train_mask]
        )

        loss.backward()
        optimizer.step()

        return loss.item()

    def evaluate(mask):
        model.eval()

        with torch.no_grad():
            out = forward_model()
            pred = out.argmax(dim=1)

            correct = (
                pred[mask] == data.y[mask]
            ).sum().item()

            total = mask.sum().item()

        return correct / total

    for epoch in range(1, 201):

        loss = train()

        if epoch % 20 == 0:
            train_acc = evaluate(data.train_mask)
            val_acc = evaluate(data.val_mask)

            print(
                f"Epoch: {epoch:03d}, "
                f"Loss: {loss:.4f}, "
                f"Train Acc: {train_acc:.4f}, "
                f"Val Acc: {val_acc:.4f}"
            )

    model.eval()

    with torch.no_grad():
        out = forward_model()
        probabilities = torch.softmax(out, dim=1)
        pred = out.argmax(dim=1)

    test_y = data.y[data.test_mask].cpu().numpy()
    test_pred = pred[data.test_mask].cpu().numpy()
    test_prob = probabilities[data.test_mask, 1].cpu().numpy()

    print("\nConfusion Matrix:")
    print(confusion_matrix(test_y, test_pred))

    print("\nClassification Report:")
    print(classification_report(
        test_y,
        test_pred,
        target_names=["Normal", "Criminal"],
        zero_division=0
    ))

    print("\nAdditional Evaluation Metrics:")

    print("Accuracy:",
          accuracy_score(test_y, test_pred))

    print("Macro Precision:",
          precision_score(test_y, test_pred, average="macro", zero_division=0))

    print("Macro Recall:",
          recall_score(test_y, test_pred, average="macro", zero_division=0))

    print("Macro F1-score:",
          f1_score(test_y, test_pred, average="macro", zero_division=0))

    print("AUC:",
          roc_auc_score(test_y, test_prob))

    print("Matthews Correlation Coefficient (MCC):",
          matthews_corrcoef(test_y, test_pred))

GCN

In [98]:
def reset_seed(seed=42):
    import random
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

reset_seed(42)

In [99]:
reset_seed(42)

gcn_a1_topic = GCN(
    num_features=data_a1_topic.num_node_features,
    hidden_channels=64
)

train_and_evaluate_updated(
    model=gcn_a1_topic,
    data=data_a1_topic,
    model_name="GCN on 1-Hop Graph with BERTopic Features",
    use_edge_attr=False,
    class_weight_1=100.0,
    lr=0.01
)


GCN on 1-Hop Graph with BERTopic Features
GCN(
  (conv1): GCNConv(472, 64)
  (conv2): GCNConv(64, 2)
)
Epoch: 020, Loss: 3.6943, Train Acc: 0.2876, Val Acc: 0.2762
Epoch: 040, Loss: 4.9720, Train Acc: 0.2225, Val Acc: 0.2173
Epoch: 060, Loss: 1.1503, Train Acc: 0.6005, Val Acc: 0.6188
Epoch: 080, Loss: 0.3857, Train Acc: 0.8021, Val Acc: 0.7956
Epoch: 100, Loss: 0.2874, Train Acc: 0.8998, Val Acc: 0.9006
Epoch: 120, Loss: 0.2714, Train Acc: 0.8801, Val Acc: 0.8748
Epoch: 140, Loss: 0.2195, Train Acc: 0.9256, Val Acc: 0.9374
Epoch: 160, Loss: 0.1887, Train Acc: 0.9416, Val Acc: 0.9466
Epoch: 180, Loss: 0.7866, Train Acc: 0.7271, Val Acc: 0.7127
Epoch: 200, Loss: 0.4223, Train Acc: 0.8082, Val Acc: 0.8177

Confusion Matrix:
[[431 107]
 [  0   5]]

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      0.80      0.89       538
    Criminal       0.04      1.00      0.09         5

    accuracy                           0.80       543
  

GRAPH SAGE

In [100]:
reset_seed(42)

graphsage_a1_topic = GraphSAGE(
    num_features=data_a1_topic.num_node_features,
    hidden_channels=64
)

train_and_evaluate_updated(
    model=graphsage_a1_topic,
    data=data_a1_topic,
    model_name="GraphSAGE on 1-Hop Graph with BERTopic Features",
    use_edge_attr=False,
    class_weight_1=100.0,
    lr=0.01
)


GraphSAGE on 1-Hop Graph with BERTopic Features
GraphSAGE(
  (conv1): SAGEConv(472, 64, aggr=mean)
  (conv2): SAGEConv(64, 2, aggr=mean)
)
Epoch: 020, Loss: 1.8841, Train Acc: 0.9582, Val Acc: 0.9576
Epoch: 040, Loss: 0.3374, Train Acc: 0.9189, Val Acc: 0.9227
Epoch: 060, Loss: 0.1783, Train Acc: 0.9324, Val Acc: 0.9263
Epoch: 080, Loss: 0.1067, Train Acc: 0.9478, Val Acc: 0.9374
Epoch: 100, Loss: 0.0555, Train Acc: 0.9625, Val Acc: 0.9521
Epoch: 120, Loss: 0.0465, Train Acc: 0.9693, Val Acc: 0.9595
Epoch: 140, Loss: 0.0412, Train Acc: 0.9717, Val Acc: 0.9613
Epoch: 160, Loss: 0.0364, Train Acc: 0.9736, Val Acc: 0.9650
Epoch: 180, Loss: 0.0338, Train Acc: 0.9779, Val Acc: 0.9705
Epoch: 200, Loss: 0.0321, Train Acc: 0.9791, Val Acc: 0.9724

Confusion Matrix:
[[524  14]
 [  0   5]]

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      0.97      0.99       538
    Criminal       0.26      1.00      0.42         5

    accuracy        

GAT

In [101]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data

def create_pyg_data_with_onehot_edge_features(graph, feature_df, topic_df, num_topics=10):

    node_list = list(graph.nodes())

    node_to_idx = {
        node: idx
        for idx, node in enumerate(node_list)
    }

    feature_cols = [
        col for col in feature_df.columns
        if col not in ["Node", "Label"]
    ]

    X_df = (
        feature_df
        .set_index("Node")
        .loc[node_list, feature_cols]
    )

    X = torch.tensor(
        X_df.values,
        dtype=torch.float
    )

    y_df = (
        feature_df
        .set_index("Node")
        .loc[node_list, "Label"]
    )

    y = torch.tensor(
        y_df.values,
        dtype=torch.long
    )

    topic_lookup = (
        topic_df
        .groupby(["From_Name", "To_Name"])["Dominant_Topic"]
        .agg(lambda x: x.mode()[0])
        .to_dict()
    )

    edge_index = []
    edge_topic_ids = []

    for source, target in graph.edges():

        edge_index.append([
            node_to_idx[source],
            node_to_idx[target]
        ])

        topic = topic_lookup.get(
            (source, target),
            0
        )

        edge_topic_ids.append(int(topic))

    edge_index = torch.tensor(
        edge_index,
        dtype=torch.long
    ).t().contiguous()

    edge_topic_ids = torch.tensor(
        edge_topic_ids,
        dtype=torch.long
    )

    edge_attr = F.one_hot(
        edge_topic_ids,
        num_classes=num_topics
    ).float()

    data = Data(
        x=X,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y
    )

    return data

In [102]:
data_a1_topic_onehot = create_pyg_data_with_onehot_edge_features(
    graph=G_a1,
    feature_df=node_features_final_a1_topic,
    topic_df=topic_df,
    num_topics=10
)

data_a1_topic_onehot = create_masks(data_a1_topic_onehot)

print(data_a1_topic_onehot)

Data(x=[2713, 472], edge_index=[2, 19197], edge_attr=[19197, 10], y=[2713], train_mask=[2713], val_mask=[2713], test_mask=[2713])


In [103]:
class GAT_Edge_OneHot(torch.nn.Module):

    def __init__(self, num_features, hidden_channels, heads, edge_dim):
        super().__init__()

        self.conv1 = GATConv(
            num_features,
            hidden_channels,
            heads=heads,
            dropout=0.3,
            edge_dim=edge_dim
        )

        self.conv2 = GATConv(
            hidden_channels * heads,
            2,
            heads=1,
            dropout=0.3,
            edge_dim=edge_dim
        )

    def forward(self, x, edge_index, edge_attr):

        x = F.dropout(x, p=0.3, training=self.training)

        x = self.conv1(
            x,
            edge_index,
            edge_attr
        )

        x = F.elu(x)

        x = F.dropout(x, p=0.3, training=self.training)

        x = self.conv2(
            x,
            edge_index,
            edge_attr
        )

        return x

In [104]:
reset_seed(42)

gat_a1_topic_onehot = GAT_Edge_OneHot(
    num_features=data_a1_topic_onehot.num_node_features,
    hidden_channels=64,
    heads=4,
    edge_dim=10
)

train_and_evaluate_updated(
    model=gat_a1_topic_onehot,
    data=data_a1_topic_onehot,
    model_name="GAT on 1-Hop Graph with One-Hot BERTopic Edge Features",
    use_edge_attr=True,
    class_weight_1=100.0,
    lr=0.001
)


GAT on 1-Hop Graph with One-Hot BERTopic Edge Features
GAT_Edge_OneHot(
  (conv1): GATConv(472, 64, heads=4)
  (conv2): GATConv(256, 2, heads=1)
)
Epoch: 020, Loss: 147.0715, Train Acc: 0.0246, Val Acc: 0.0313
Epoch: 040, Loss: 10.2419, Train Acc: 0.1401, Val Acc: 0.1234
Epoch: 060, Loss: 11.4886, Train Acc: 0.1709, Val Acc: 0.1436
Epoch: 080, Loss: 12.1209, Train Acc: 0.1315, Val Acc: 0.0939
Epoch: 100, Loss: 12.5327, Train Acc: 0.1862, Val Acc: 0.1565
Epoch: 120, Loss: 19.0663, Train Acc: 0.1623, Val Acc: 0.1252
Epoch: 140, Loss: 8.7525, Train Acc: 0.1905, Val Acc: 0.1565
Epoch: 160, Loss: 8.8387, Train Acc: 0.2624, Val Acc: 0.2468
Epoch: 180, Loss: 10.4224, Train Acc: 0.1991, Val Acc: 0.1584
Epoch: 200, Loss: 10.2726, Train Acc: 0.1850, Val Acc: 0.1455

Confusion Matrix:
[[ 90 448]
 [  0   5]]

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      0.17      0.29       538
    Criminal       0.01      1.00      0.02         5

   

TAGCN

In [105]:
reset_seed(42)

tagcn_a3_topic = TAGCN(
    num_features=data_a3_topic.num_node_features,
    hidden_channels=64
)

train_and_evaluate_updated(
    model=tagcn_a3_topic,
    data=data_a3_topic,
    model_name="TAGCN on 3-Hop Graph with BERTopic Features",
    use_edge_attr=False,
    class_weight_1=100.0,
    lr=0.01
)


TAGCN on 3-Hop Graph with BERTopic Features
TAGCN(
  (conv1): TAGConv(472, 64, K=3)
  (conv2): TAGConv(64, 2, K=3)
)
Epoch: 020, Loss: 0.2876, Train Acc: 0.9909, Val Acc: 0.9922
Epoch: 040, Loss: 0.2672, Train Acc: 0.9934, Val Acc: 0.9937
Epoch: 060, Loss: 0.0910, Train Acc: 0.9945, Val Acc: 0.9944
Epoch: 080, Loss: 0.0205, Train Acc: 0.9972, Val Acc: 0.9966
Epoch: 100, Loss: 0.0147, Train Acc: 0.9980, Val Acc: 0.9967
Epoch: 120, Loss: 0.0122, Train Acc: 0.9986, Val Acc: 0.9973
Epoch: 140, Loss: 0.0110, Train Acc: 0.9987, Val Acc: 0.9976
Epoch: 160, Loss: 0.0102, Train Acc: 0.9987, Val Acc: 0.9975
Epoch: 180, Loss: 0.0096, Train Acc: 0.9987, Val Acc: 0.9975
Epoch: 200, Loss: 0.0091, Train Acc: 0.9987, Val Acc: 0.9975

Confusion Matrix:
[[6361   12]
 [   0    5]]

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      6373
    Criminal       0.29      1.00      0.45         5

    accuracy                          

KNN

In [106]:
reset_seed(42)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

# =====================================================
# KNN using 1-Hop Feature Table with BERTopic Features
# =====================================================

knn_data_a1_topic = node_features_final_a1_topic.copy()

X_knn_a1_topic = knn_data_a1_topic.drop(
    columns=["Node", "Label"]
)

y_knn_a1_topic = knn_data_a1_topic["Label"]

print("X shape:", X_knn_a1_topic.shape)
print("y shape:", y_knn_a1_topic.shape)

print("\nLabel count:")
print(y_knn_a1_topic.value_counts())

# =====================================================
# Train-Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_knn_a1_topic,
    y_knn_a1_topic,
    test_size=0.2,
    random_state=42,
    stratify=y_knn_a1_topic
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

print("\nTrain Labels:")
print(y_train.value_counts())

print("\nTest Labels:")
print(y_test.value_counts())

# =====================================================
# Feature Scaling
# =====================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================================================
# KNN Model
# =====================================================

knn_a1_topic = KNeighborsClassifier(
    n_neighbors=3
)

knn_a1_topic.fit(
    X_train_scaled,
    y_train
)

print("\nKNN model trained successfully")

# =====================================================
# Final Evaluation
# =====================================================

y_pred = knn_a1_topic.predict(X_test_scaled)
y_prob = knn_a1_topic.predict_proba(X_test_scaled)[:, 1]

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Criminal"],
    zero_division=0
))

print("\nAdditional Evaluation Metrics:")

print("Accuracy:",
      accuracy_score(y_test, y_pred))

print("Macro Precision:",
      precision_score(y_test, y_pred, average="macro", zero_division=0))

print("Macro Recall:",
      recall_score(y_test, y_pred, average="macro", zero_division=0))

print("Macro F1-score:",
      f1_score(y_test, y_pred, average="macro", zero_division=0))

print("AUC:",
      roc_auc_score(y_test, y_prob))

print("Matthews Correlation Coefficient (MCC):",
      matthews_corrcoef(y_test, y_pred))

X shape: (2713, 472)
y shape: (2713,)

Label count:
Label
0    2688
1      25
Name: count, dtype: int64

Train Shape: (2170, 472)
Test Shape: (543, 472)

Train Labels:
Label
0    2150
1      20
Name: count, dtype: int64

Test Labels:
Label
0    538
1      5
Name: count, dtype: int64

KNN model trained successfully

Confusion Matrix:
[[538   0]
 [  4   1]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00       538
    Criminal       1.00      0.20      0.33         5

    accuracy                           0.99       543
   macro avg       1.00      0.60      0.66       543
weighted avg       0.99      0.99      0.99       543


Additional Evaluation Metrics:
Accuracy: 0.992633517495396
Macro Precision: 0.996309963099631
Macro Recall: 0.6
Macro F1-score: 0.6648148148148149
AUC: 0.6
Matthews Correlation Coefficient (MCC): 0.4455603048296071


**TAGCN+COMMUNITY DETECTION+BERTOPIC**

lovian

In [108]:
# =====================================================
# Community-wise Topic Analysis
# =====================================================

import pandas as pd

# -----------------------------------------------------
# Step 1: Map sender community
# -----------------------------------------------------

topic_community_df = topic_df.copy()

topic_community_df = topic_community_df.merge(
    community_df,
    left_on="From_Name",
    right_on="Node",
    how="left"
)

topic_community_df = topic_community_df.rename(
    columns={"Community": "From_Community"}
)

topic_community_df = topic_community_df.drop(columns=["Node"])

# -----------------------------------------------------
# Step 2: Map receiver community
# -----------------------------------------------------

topic_community_df = topic_community_df.merge(
    community_df,
    left_on="To_Name",
    right_on="Node",
    how="left"
)

topic_community_df = topic_community_df.rename(
    columns={"Community": "To_Community"}
)

topic_community_df = topic_community_df.drop(columns=["Node"])

print("Topic-community dataframe shape:")
print(topic_community_df.shape)

print("\nMissing community values:")
print(topic_community_df[["From_Community", "To_Community"]].isnull().sum())


# =====================================================
# Step 3: Keep emails where sender and receiver are in same community
# =====================================================

same_community_emails = topic_community_df[
    topic_community_df["From_Community"] == topic_community_df["To_Community"]
].copy()

same_community_emails = same_community_emails.rename(
    columns={"From_Community": "Community"}
)

print("\nEmails inside same community:")
print(same_community_emails.shape)


# =====================================================
# Step 4: Count topic distribution inside each community
# =====================================================

community_topic_counts = (
    same_community_emails
    .groupby(["Community", "Topic"])
    .size()
    .reset_index(name="Topic_Email_Count")
)

print("\nCommunity-topic counts:")
print(community_topic_counts.head())


# =====================================================
# Step 5: Find dominant topic per community
# =====================================================

dominant_topic_per_community = (
    community_topic_counts
    .sort_values(["Community", "Topic_Email_Count"], ascending=[True, False])
    .groupby("Community")
    .head(1)
    .reset_index(drop=True)
)

dominant_topic_per_community = dominant_topic_per_community.rename(
    columns={
        "Topic": "Dominant_Topic",
        "Topic_Email_Count": "Dominant_Topic_Email_Count"
    }
)

print("\nDominant Topic per Community:")
print(dominant_topic_per_community.head(20))


# =====================================================
# Step 6: Add community size and criminal count
# =====================================================

community_topic_summary = community_summary.merge(
    dominant_topic_per_community,
    on="Community",
    how="left"
)

print("\nCommunity Topic Summary:")
print(community_topic_summary.head(20))


# =====================================================
# Step 7: Focus only on suspicious communities
# =====================================================

suspicious_community_topics = community_topic_summary[
    community_topic_summary["Criminal_Count"] >= 2
].copy()

print("\nSuspicious Communities with Dominant Topics:")
print(suspicious_community_topics)


# =====================================================
# Step 8: Save output
# =====================================================

community_topic_summary.to_csv(
    "community_topic_summary.csv",
    index=False
)

suspicious_community_topics.to_csv(
    "suspicious_community_topics.csv",
    index=False
)

print("\nSaved:")
print("community_topic_summary.csv")
print("suspicious_community_topics.csv")

Topic-community dataframe shape:
(327696, 10)

Missing community values:
From_Community    7816
To_Community      8345
dtype: int64

Emails inside same community:
(248910, 10)

Community-topic counts:
   Community  Topic  Topic_Email_Count
0        0.0      0                 40
1        0.0      1                 83
2        0.0      2                156
3        0.0      3               1135
4        0.0      6                  1

Dominant Topic per Community:
    Community  Dominant_Topic  Dominant_Topic_Email_Count
0         0.0               3                        1135
1         1.0               0                         484
2         2.0               0                        5225
3         3.0               0                        3693
4         4.0               3                          15
5         5.0               3                         771
6         6.0               2                        4740
7         7.0               1                        5018
8         8.

NameError: name 'community_summary' is not defined

In [109]:
print(community_df.head())
print(criminal_community_df.head())

                      Node  Community
0               <8=0         38
1                  scorman         38
2                      " "         40
3               kate symes         40
4  " - inside nytimes.com"          2
                    Node  Community
8       jeffery skilling         18
55           kenneth lay         18
959   calger christopher         18
1107           ray bowen         18
1343      timothy belden         40


In [110]:
# Community Size

community_sizes = (
    community_df["Community"]
    .value_counts()
    .reset_index()
)

community_sizes.columns = [
    "Community",
    "Community_Size"
]

# Criminal Count

criminal_counts = (
    criminal_community_df["Community"]
    .value_counts()
    .reset_index()
)

criminal_counts.columns = [
    "Community",
    "Criminal_Count"
]

# Community Summary

community_summary = community_sizes.merge(
    criminal_counts,
    on="Community",
    how="left"
)

community_summary["Criminal_Count"] = (
    community_summary["Criminal_Count"]
    .fillna(0)
    .astype(int)
)

community_summary.head()

,Community,Community_Size,Criminal_Count
0,18,4093,21
1,8,4082,0
2,38,3152,1
3,12,3129,0
4,15,2506,0


In [111]:
community_topic_summary = community_summary.merge(
    dominant_topic_per_community,
    on="Community",
    how="left"
)

community_topic_summary.head(20)


,Community,Community_Size,Criminal_Count,Dominant_Topic,Dominant_Topic_Email_Count
0,18,4093,21,2,7467
1,8,4082,0,0,24149
2,38,3152,1,0,10994
3,12,3129,0,1,8468
4,15,2506,0,0,8748
5,13,2242,1,1,4858
6,6,1816,0,2,4740
7,40,1459,2,0,7260
8,16,1376,0,1,4150
9,0,1292,0,3,1135


In [112]:
community_topic_counts[
    community_topic_counts["Community"] == 18
].sort_values(
    "Topic_Email_Count",
    ascending=False
)

,Community,Topic,Topic_Email_Count
156,18.0,2,7467
154,18.0,0,6921
155,18.0,1,6256
157,18.0,3,1788
160,18.0,6,1675
158,18.0,4,1313
163,18.0,9,991
162,18.0,8,794
161,18.0,7,704
159,18.0,5,698


spectral clustering 


In [115]:
# =====================================================
# Spectral Clustering Analysis with K = 30
# =====================================================

import pandas as pd
import numpy as np
import networkx as nx

from sklearn.cluster import SpectralClustering
from sklearn.metrics import normalized_mutual_info_score
from networkx.algorithms.community.quality import modularity


# =====================================================
# Step 1: Set Final K Value
# =====================================================

K = 30

print("Final number of clusters selected:", K)


# =====================================================
# Step 2: Prepare Graph for Evaluation
# =====================================================
# Spectral clustering is performed on node features.
# Modularity is evaluated using the graph structure.

G_eval = G_a3.to_undirected()

print("Graph nodes:", G_eval.number_of_nodes())
print("Graph edges:", G_eval.number_of_edges())


# =====================================================
# Step 3: True Labels
# =====================================================

true_labels = spectral_data["Label"].values

print("\nLabel count:")
print(pd.Series(true_labels).value_counts())


# =====================================================
# Step 4: Apply Spectral Clustering
# =====================================================

spectral = SpectralClustering(
    n_clusters=K,
    affinity="nearest_neighbors",
    n_neighbors=10,
    assign_labels="kmeans",
    random_state=42
)

cluster_labels = spectral.fit_predict(
    X_spectral_scaled
)

spectral_data_clustered = spectral_data.copy()
spectral_data_clustered["Cluster"] = cluster_labels

print("\nSpectral clustering completed.")
print("Clustered data shape:", spectral_data_clustered.shape)

print("\nCluster label count:")
print(
    spectral_data_clustered["Cluster"]
    .value_counts()
    .sort_index()
)


# =====================================================
# Step 5: NMI Score
# =====================================================
# NMI checks how much the clusters agree with actual labels.
# Since clustering is unsupervised, low NMI is common.

nmi_score = normalized_mutual_info_score(
    true_labels,
    cluster_labels
)

print("\nNMI Score:", nmi_score)


# =====================================================
# Step 6: Create Cluster Communities for Modularity
# =====================================================

communities = []

for cluster_id in sorted(spectral_data_clustered["Cluster"].unique()):

    cluster_nodes = set(
        spectral_data_clustered[
            spectral_data_clustered["Cluster"] == cluster_id
        ]["Node"]
    )

    communities.append(cluster_nodes)


# =====================================================
# Step 7: Modularity Score
# =====================================================
# Modularity checks whether clusters are meaningful in graph structure.

modularity_score = modularity(
    G_eval,
    communities,
    weight="weight"
)

print("Modularity Score:", modularity_score)


# =====================================================
# Step 8: Cluster Size Analysis
# =====================================================

cluster_sizes = (
    spectral_data_clustered["Cluster"]
    .value_counts()
    .reset_index()
)

cluster_sizes.columns = [
    "Cluster",
    "Cluster_Size"
]

cluster_sizes = cluster_sizes.sort_values(
    "Cluster_Size",
    ascending=False
)

print("\nTop 20 Largest Clusters:")
print(cluster_sizes.head(20))


# =====================================================
# Step 9: Criminal Count per Cluster
# =====================================================

criminal_cluster_counts = (
    spectral_data_clustered[
        spectral_data_clustered["Label"] == 1
    ]
    .groupby("Cluster")
    .size()
    .reset_index(name="Criminal_Count")
)

print("\nCriminal Count per Cluster:")
print(
    criminal_cluster_counts
    .sort_values("Criminal_Count", ascending=False)
)


# =====================================================
# Step 10: Cluster Summary
# =====================================================

spectral_cluster_summary = cluster_sizes.merge(
    criminal_cluster_counts,
    on="Cluster",
    how="left"
)

spectral_cluster_summary["Criminal_Count"] = (
    spectral_cluster_summary["Criminal_Count"]
    .fillna(0)
    .astype(int)
)

spectral_cluster_summary["Criminal_Ratio"] = (
    spectral_cluster_summary["Criminal_Count"] /
    spectral_cluster_summary["Cluster_Size"]
)

spectral_cluster_summary = spectral_cluster_summary.sort_values(
    ["Criminal_Count", "Criminal_Ratio"],
    ascending=False
)

print("\nSpectral Cluster Summary:")
print(spectral_cluster_summary.head(20))


# =====================================================
# Step 11: Suspicious Clusters
# =====================================================
# Suspicious cluster = cluster with 2 or more criminal nodes.

suspicious_spectral_clusters = spectral_cluster_summary[
    spectral_cluster_summary["Criminal_Count"] >= 2
].copy()

print("\nSuspicious Spectral Clusters:")
print(suspicious_spectral_clusters)


# =====================================================
# Step 12: Largest Criminal Cluster
# =====================================================

largest_criminal_cluster = (
    spectral_cluster_summary["Criminal_Count"].max()
)

suspicious_cluster_count = (
    suspicious_spectral_clusters.shape[0]
)

print("\nLargest Criminal Cluster:", largest_criminal_cluster)
print("Suspicious Cluster Count:", suspicious_cluster_count)


# =====================================================
# Step 13: Final Evaluation Table
# =====================================================

spectral_final_result = pd.DataFrame({
    "K": [K],
    "NMI": [nmi_score],
    "Modularity": [modularity_score],
    "Largest_Criminal_Cluster": [largest_criminal_cluster],
    "Suspicious_Cluster_Count": [suspicious_cluster_count]
})

print("\nFinal Spectral Clustering Result:")
print(spectral_final_result)


# =====================================================
# Step 14: Save Outputs
# =====================================================

spectral_data_clustered.to_csv(
    "spectral_data_clustered_k30.csv",
    index=False
)

spectral_cluster_summary.to_csv(
    "spectral_cluster_summary_k30.csv",
    index=False
)

suspicious_spectral_clusters.to_csv(
    "suspicious_spectral_clusters_k30.csv",
    index=False
)

spectral_final_result.to_csv(
    "spectral_final_result_k30.csv",
    index=False
)

print("\nSaved:")
print("spectral_data_clustered_k30.csv")
print("spectral_cluster_summary_k30.csv")
print("suspicious_spectral_clusters_k30.csv")
print("spectral_final_result_k30.csv")

Final number of clusters selected: 30
Graph nodes: 31887
Graph edges: 86694

Label count:
0    31862
1       25
Name: count, dtype: int64

Spectral clustering completed.
Clustered data shape: (31887, 81)

Cluster label count:
Cluster
0     2040
1      527
2     8819
3     3609
4     1551
5      680
6     1223
7       23
8       16
9      117
10     132
11     109
12     346
13      43
14      20
15      72
16    5820
17     648
18    2793
19      13
20      45
21     527
22     410
23     631
24     463
25      61
26     157
27     212
28      58
29     722
Name: count, dtype: int64

NMI Score: 0.0009476547301692264
Modularity Score: 0.31209591814627763

Top 20 Largest Clusters:
    Cluster  Cluster_Size
0         2          8819
1        16          5820
2         3          3609
3        18          2793
4         0          2040
5         4          1551
6         6          1223
7        29           722
8         5           680
9        17           648
10       23           631


In [114]:
# =====================================================
# Spectral Cluster + BERTopic Integration
# =====================================================

# spectral_data_clustered should contain:
# Node, Label, Cluster

# topic_df should contain:
# From_Name, To_Name, Topic, Label

spectral_topic_df = topic_df.copy()

# -----------------------------------------------------
# Step 1: Map sender spectral cluster
# -----------------------------------------------------

spectral_topic_df = spectral_topic_df.merge(
    spectral_data_clustered[["Node", "Cluster"]],
    left_on="From_Name",
    right_on="Node",
    how="left"
)

spectral_topic_df = spectral_topic_df.rename(
    columns={"Cluster": "From_Cluster"}
)

spectral_topic_df = spectral_topic_df.drop(columns=["Node"])


# -----------------------------------------------------
# Step 2: Map receiver spectral cluster
# -----------------------------------------------------

spectral_topic_df = spectral_topic_df.merge(
    spectral_data_clustered[["Node", "Cluster"]],
    left_on="To_Name",
    right_on="Node",
    how="left"
)

spectral_topic_df = spectral_topic_df.rename(
    columns={"Cluster": "To_Cluster"}
)

spectral_topic_df = spectral_topic_df.drop(columns=["Node"])


print("Spectral-topic dataframe shape:")
print(spectral_topic_df.shape)

print("\nMissing cluster values:")
print(spectral_topic_df[["From_Cluster", "To_Cluster"]].isnull().sum())


# =====================================================
# Step 3: Keep emails where sender and receiver are in same spectral cluster
# =====================================================

same_cluster_emails = spectral_topic_df[
    spectral_topic_df["From_Cluster"] == spectral_topic_df["To_Cluster"]
].copy()

same_cluster_emails = same_cluster_emails.rename(
    columns={"From_Cluster": "Cluster"}
)

print("\nEmails inside same spectral cluster:")
print(same_cluster_emails.shape)


# =====================================================
# Step 4: Topic distribution inside each spectral cluster
# =====================================================

spectral_cluster_topic_counts = (
    same_cluster_emails
    .groupby(["Cluster", "Topic"])
    .size()
    .reset_index(name="Topic_Email_Count")
)

print("\nSpectral Cluster Topic Counts:")
print(spectral_cluster_topic_counts.head())


# =====================================================
# Step 5: Dominant topic per spectral cluster
# =====================================================

dominant_topic_per_spectral_cluster = (
    spectral_cluster_topic_counts
    .sort_values(["Cluster", "Topic_Email_Count"], ascending=[True, False])
    .groupby("Cluster")
    .head(1)
    .reset_index(drop=True)
)

dominant_topic_per_spectral_cluster = dominant_topic_per_spectral_cluster.rename(
    columns={
        "Topic": "Dominant_Topic",
        "Topic_Email_Count": "Dominant_Topic_Email_Count"
    }
)

print("\nDominant Topic per Spectral Cluster:")
print(dominant_topic_per_spectral_cluster.head(20))


# =====================================================
# Step 6: Merge with spectral cluster summary
# =====================================================

spectral_cluster_topic_summary = spectral_cluster_summary.merge(
    dominant_topic_per_spectral_cluster,
    on="Cluster",
    how="left"
)

print("\nSpectral Cluster Topic Summary:")
print(spectral_cluster_topic_summary.head(20))


# =====================================================
# Step 7: Suspicious spectral clusters with dominant topics
# =====================================================

suspicious_spectral_cluster_topics = spectral_cluster_topic_summary[
    spectral_cluster_topic_summary["Criminal_Count"] >= 2
].copy()

print("\nSuspicious Spectral Clusters with Dominant Topics:")
print(suspicious_spectral_cluster_topics)


# =====================================================
# Step 8: Top topics inside most suspicious cluster
# =====================================================

most_suspicious_cluster = suspicious_spectral_cluster_topics.iloc[0]["Cluster"]

top_topics_most_suspicious_cluster = (
    spectral_cluster_topic_counts[
        spectral_cluster_topic_counts["Cluster"] == most_suspicious_cluster
    ]
    .sort_values("Topic_Email_Count", ascending=False)
)

print("\nTop Topics in Most Suspicious Spectral Cluster:")
print(top_topics_most_suspicious_cluster)


# =====================================================
# Step 9: Save outputs
# =====================================================

spectral_cluster_topic_summary.to_csv(
    "spectral_cluster_topic_summary.csv",
    index=False
)

suspicious_spectral_cluster_topics.to_csv(
    "suspicious_spectral_cluster_topics.csv",
    index=False
)

top_topics_most_suspicious_cluster.to_csv(
    "top_topics_most_suspicious_spectral_cluster.csv",
    index=False
)

print("\nSaved:")
print("spectral_cluster_topic_summary.csv")
print("suspicious_spectral_cluster_topics.csv")
print("top_topics_most_suspicious_spectral_cluster.csv")

Spectral-topic dataframe shape:
(327696, 10)

Missing cluster values:
From_Cluster    7816
To_Cluster      8345
dtype: int64

Emails inside same spectral cluster:
(170888, 10)

Spectral Cluster Topic Counts:
   Cluster  Topic  Topic_Email_Count
0      0.0      0               2826
1      0.0      1                654
2      0.0      2                338
3      0.0      3                 99
4      0.0      4                 56

Dominant Topic per Spectral Cluster:
    Cluster  Dominant_Topic  Dominant_Topic_Email_Count
0       0.0               0                        2826
1       1.0               1                           7
2       2.0               0                        1484
3       3.0               2                           3
4       4.0               0                         610
5       5.0               0                        5429
6       6.0               1                           1
7      16.0               0                       25820
8      17.0               1 